In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import gc

SEED = 42
np.random.seed(SEED)

ROOT = Path(_hcc_path('LIHC_scRNA_Step19_QC'))

DATA_DIR = ROOT / "Data"
NOTEBOOK_DIR = ROOT / "Notebook"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"

for p in [
    DATA_DIR,
    NOTEBOOK_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = Path(
    _hcc_path('LIHC_scRNA_Step18_GSE149614/Results/GSE149614_RAW_UNFILTERED.h5ad')
)

print("Input:")
print(INPUT_H5AD)
print("Exists:", INPUT_H5AD.exists())

assert INPUT_H5AD.exists()

print("\nSTEP19 INITIALIZED")

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)

print(adata)

print("\nShape:", adata.shape)
print("Cells:", f"{adata.n_obs:,}")
print("Genes:", f"{adata.n_vars:,}")

print("\nMetadata:")
print(adata.obs.columns.tolist())

In [ ]:
print("=" * 80)
print("RAW COUNT CHECK")
print("=" * 80)

print("Matrix type:")
print(type(adata.X))

print("\nMinimum:")
print(adata.X.min())

print("\nMaximum:")
print(adata.X.max())

sample_values = adata.X[:1000, :1000]

if hasattr(sample_values, "data"):
    sampled = sample_values.data
else:
    sampled = np.asarray(sample_values).ravel()

if len(sampled) > 0:
    integer_fraction = np.mean(
        np.isclose(
            sampled,
            np.round(sampled)
        )
    )
else:
    integer_fraction = 1.0

print("\nApprox integer fraction:")
print(integer_fraction)

assert adata.X.min() >= 0
assert integer_fraction > 0.99

print("\nRAW COUNTS CHECK = PASS")

In [ ]:
gene_names_upper = pd.Index(
    adata.var_names.astype(str)
).str.upper()

adata.var["mt"] = gene_names_upper.str.startswith("MT-")

adata.var["ribo"] = (
    gene_names_upper.str.startswith("RPS")
    |
    gene_names_upper.str.startswith("RPL")
)

print("Mitochondrial genes:",
      int(adata.var["mt"].sum()))

print("Ribosomal genes:",
      int(adata.var["ribo"].sum()))

print("\nExample mitochondrial genes:")
print(
    adata.var_names[
        adata.var["mt"]
    ][:20].tolist()
)

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo"],
    percent_top=None,
    log1p=False,
    inplace=True
)

qc_cols = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo"
]

display(
    adata.obs[
        qc_cols
    ].head()
)

In [ ]:
qc_summary = (
    adata.obs[
        [
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
            "pct_counts_ribo"
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)

display(qc_summary)

qc_summary.to_csv(
    RESULTS_DIR /
    "QC_Global_Summary.csv",
    encoding="utf-8-sig"
)

In [ ]:
sample_qc = (
    adata.obs
    .groupby(
        "sample",
        observed=True
    )
    .agg(
        N_cells=("sample", "size"),

        median_genes=(
            "n_genes_by_counts",
            "median"
        ),

        median_counts=(
            "total_counts",
            "median"
        ),

        median_mt=(
            "pct_counts_mt",
            "median"
        ),

        p95_mt=(
            "pct_counts_mt",
            lambda x:
                np.percentile(x, 95)
        )
    )
    .reset_index()
)

display(sample_qc)

sample_qc.to_csv(
    RESULTS_DIR /
    "QC_By_Sample_Before_Filtering.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
patient_qc = (
    adata.obs
    .groupby(
        "patient",
        observed=True
    )
    .agg(
        N_cells=("patient", "size"),

        median_genes=(
            "n_genes_by_counts",
            "median"
        ),

        median_counts=(
            "total_counts",
            "median"
        ),

        median_mt=(
            "pct_counts_mt",
            "median"
        )
    )
    .reset_index()
)

display(patient_qc)

patient_qc.to_csv(
    RESULTS_DIR /
    "QC_By_Patient_Before_Filtering.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    adata.obs["n_genes_by_counts"],
    bins=100
)

ax.set_xlabel("Genes detected per cell")
ax.set_ylabel("Number of cells")
ax.set_title("Distribution of detected genes")

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "QC_nGenes_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
log_counts = np.log10(
    adata.obs["total_counts"] + 1
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.hist(
    log_counts,
    bins=100
)

ax.set_xlabel("log10(total counts + 1)")
ax.set_ylabel("Number of cells")
ax.set_title("Distribution of total UMI counts")

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "QC_TotalCounts_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.hist(
    adata.obs["pct_counts_mt"],
    bins=100
)

ax.set_xlabel("Mitochondrial reads (%)")
ax.set_ylabel("Number of cells")
ax.set_title("Distribution of mitochondrial fraction")

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "QC_Mitochondrial_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
plot_n = min(
    30000,
    adata.n_obs
)

rng = np.random.default_rng(SEED)

plot_idx = rng.choice(
    adata.n_obs,
    size=plot_n,
    replace=False
)

plot_obs = (
    adata.obs
    .iloc[plot_idx]
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

ax.scatter(
    np.log10(
        plot_obs["total_counts"] + 1
    ),
    plot_obs["n_genes_by_counts"],
    s=4,
    alpha=0.25
)

ax.set_xlabel(
    "log10(total counts + 1)"
)

ax.set_ylabel(
    "Genes detected"
)

ax.set_title(
    "Genes detected vs total counts"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "QC_Genes_vs_Counts.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
metrics = [
    "n_genes_by_counts",
    "total_counts",
    "pct_counts_mt",
    "pct_counts_ribo"
]

quantiles = [
    0,
    0.001,
    0.005,
    0.01,
    0.025,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.975,
    0.99,
    0.995,
    0.999,
    1
]

quantile_table = (
    adata.obs[
        metrics
    ]
    .quantile(
        quantiles
    )
)

display(quantile_table)

quantile_table.to_csv(
    RESULTS_DIR /
    "QC_Quantiles.csv",
    encoding="utf-8-sig"
)

In [ ]:
site_qc = (
    adata.obs
    .groupby(
        "site",
        observed=True
    )
    .agg(
        N_cells=("site", "size"),

        median_genes=(
            "n_genes_by_counts",
            "median"
        ),

        median_counts=(
            "total_counts",
            "median"
        ),

        median_mt=(
            "pct_counts_mt",
            "median"
        ),

        p95_mt=(
            "pct_counts_mt",
            lambda x:
                np.percentile(x, 95)
        )
    )
    .reset_index()
)

display(site_qc)

site_qc.to_csv(
    RESULTS_DIR /
    "QC_By_Site_Before_Filtering.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
qc_unfiltered_path = (
    DATA_DIR /
    "GSE149614_QC_METRICS_UNFILTERED.h5ad"
)

adata.write_h5ad(
    qc_unfiltered_path,
    compression="gzip"
)

print("Saved:")
print(qc_unfiltered_path)

In [ ]:
print("=" * 100)
print("STEP19 QC SUMMARY FOR REVIEW")
print("=" * 100)

# ============================================================
# 1. Global QC summary
# ============================================================
print("\n[1] GLOBAL QC SUMMARY")
print("-" * 100)

display(qc_summary)


# ============================================================
# 2. Sample-level QC summary
# ============================================================
print("\n[2] SAMPLE-LEVEL QC SUMMARY")
print("-" * 100)

display(sample_qc)


# ============================================================
# 3. Site-level QC summary
# ============================================================
print("\n[3] SITE-LEVEL QC SUMMARY")
print("-" * 100)

display(site_qc)


# ============================================================
# 4. Key quantiles for filtering decisions
# ============================================================
print("\n[4] KEY QC QUANTILES")
print("-" * 100)

key_metrics = [
    "n_genes_by_counts",
    "total_counts",
    "pct_counts_mt"
]

key_quantiles = [
    0,
    0.001,
    0.005,
    0.01,
    0.025,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.975,
    0.99,
    0.995,
    0.999,
    1
]

qc_key_quantiles = (
    adata.obs[key_metrics]
    .quantile(key_quantiles)
)

display(qc_key_quantiles)


# ============================================================
# 5. Extra cohort information
# ============================================================
print("\n[5] COHORT QC CONTEXT")
print("-" * 100)

print("Total cells:", f"{adata.n_obs:,}")
print("Total genes:", f"{adata.n_vars:,}")

print("\nPatients:")
print(
    adata.obs["patient"]
    .value_counts()
    .sort_index()
)

print("\nSamples:")
print(
    adata.obs["sample"]
    .value_counts()
    .sort_index()
)

print("\nSites:")
print(
    adata.obs["site"]
    .value_counts()
)


# ============================================================
# 6. Save review table
# ============================================================
qc_key_quantiles.to_csv(
    RESULTS_DIR / "QC_Key_Quantiles_For_Review.csv",
    encoding="utf-8-sig"
)

print("\nSaved:")
print(
    RESULTS_DIR /
    "QC_Key_Quantiles_For_Review.csv"
)

print("\n" + "=" * 100)
print("END OF STEP19 QC SUMMARY")
print("=" * 100)

In [ ]:
# ============================================================
# STEP19 - QC threshold sensitivity audit
# ============================================================

qc_test = adata.obs.copy()

# ------------------------------------------------------------
# Scheme A: very lenient
# ------------------------------------------------------------
qc_test["Pass_A_500_20"] = (
    (qc_test["n_genes_by_counts"] >= 500)
    &
    (qc_test["pct_counts_mt"] < 20)
)

# ------------------------------------------------------------
# Scheme B: candidate primary rule
# ------------------------------------------------------------
qc_test["Pass_B_600_15"] = (
    (qc_test["n_genes_by_counts"] >= 600)
    &
    (qc_test["pct_counts_mt"] < 15)
)

# ------------------------------------------------------------
# Scheme C: somewhat stricter
# ------------------------------------------------------------
qc_test["Pass_C_700_12"] = (
    (qc_test["n_genes_by_counts"] >= 700)
    &
    (qc_test["pct_counts_mt"] < 12)
)

print("QC candidate rules created.")

In [ ]:
# ============================================================
# Overall retention under candidate thresholds
# ============================================================

schemes = {
    "A_500genes_mt20": "Pass_A_500_20",
    "B_600genes_mt15": "Pass_B_600_15",
    "C_700genes_mt12": "Pass_C_700_12"
}

overall_results = []

for name, col in schemes.items():

    n_total = len(qc_test)
    n_keep = int(qc_test[col].sum())
    n_remove = n_total - n_keep

    overall_results.append({
        "Scheme": name,
        "Total_cells": n_total,
        "Retained_cells": n_keep,
        "Removed_cells": n_remove,
        "Retention_pct": n_keep / n_total * 100,
        "Removal_pct": n_remove / n_total * 100
    })

overall_qc_sensitivity = pd.DataFrame(
    overall_results
)

display(overall_qc_sensitivity)

overall_qc_sensitivity.to_csv(
    RESULTS_DIR /
    "QC_Threshold_Sensitivity_Overall.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# Sample-level retention under candidate thresholds
# ============================================================

sample_results = []

for sample, df in qc_test.groupby(
    "sample",
    observed=True
):

    row = {
        "sample": sample,
        "N_before": len(df)
    }

    for name, col in schemes.items():

        n_keep = int(df[col].sum())

        row[f"{name}_N"] = n_keep

        row[f"{name}_Retention_pct"] = (
            n_keep / len(df) * 100
        )

    sample_results.append(row)

sample_sensitivity = pd.DataFrame(
    sample_results
)

display(sample_sensitivity)

sample_sensitivity.to_csv(
    RESULTS_DIR /
    "QC_Threshold_Sensitivity_By_Sample.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# Site-level retention
# ============================================================

site_results = []

for site, df in qc_test.groupby(
    "site",
    observed=True
):

    row = {
        "site": site,
        "N_before": len(df)
    }

    for name, col in schemes.items():

        n_keep = int(df[col].sum())

        row[f"{name}_N"] = n_keep

        row[f"{name}_Retention_pct"] = (
            n_keep / len(df) * 100
        )

    site_results.append(row)

site_sensitivity = pd.DataFrame(
    site_results
)

display(site_sensitivity)

site_sensitivity.to_csv(
    RESULTS_DIR /
    "QC_Threshold_Sensitivity_By_Site.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# Candidate B failure reasons
# ============================================================

qc_test["Fail_low_genes_B"] = (
    qc_test["n_genes_by_counts"] < 600
)

qc_test["Fail_high_mt_B"] = (
    qc_test["pct_counts_mt"] >= 15
)

qc_test["Fail_both_B"] = (
    qc_test["Fail_low_genes_B"]
    &
    qc_test["Fail_high_mt_B"]
)

print("=" * 80)
print("CANDIDATE B FAILURE REASONS")
print("=" * 80)

print(
    "Low genes (<600):",
    int(qc_test["Fail_low_genes_B"].sum())
)

print(
    "High mt (>=15%):",
    int(qc_test["Fail_high_mt_B"].sum())
)

print(
    "Both:",
    int(qc_test["Fail_both_B"].sum())
)

print(
    "\nTotal failing candidate B:",
    int((~qc_test["Pass_B_600_15"]).sum())
)

In [ ]:
# ============================================================
# Sample-aware MAD thresholds
# ============================================================

def mad(x):
    x = np.asarray(x)
    med = np.median(x)
    return np.median(
        np.abs(x - med)
    )


sample_thresholds = []

for sample, df in adata.obs.groupby(
    "sample",
    observed=True
):

    genes = df["n_genes_by_counts"].values
    mt = df["pct_counts_mt"].values

    genes_med = np.median(genes)
    genes_mad = mad(genes)

    mt_med = np.median(mt)
    mt_mad = mad(mt)

    # Lower gene threshold:
    # sample-specific but never below 500
    lower_genes = max(
        500,
        genes_med - 4 * genes_mad
    )

    # mt threshold:
    # sample-specific upper tail
    # but never allow >20%
    upper_mt = min(
        20,
        mt_med + 4 * mt_mad
    )

    sample_thresholds.append({
        "sample": sample,
        "median_genes": genes_med,
        "MAD_genes": genes_mad,
        "lower_genes_threshold": lower_genes,
        "median_mt": mt_med,
        "MAD_mt": mt_mad,
        "upper_mt_threshold": upper_mt
    })

sample_thresholds = pd.DataFrame(
    sample_thresholds
)

display(sample_thresholds)

sample_thresholds.to_csv(
    RESULTS_DIR /
    "QC_Sample_Specific_MAD_Thresholds.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# Apply MAD rule as audit only
# ============================================================

threshold_map = (
    sample_thresholds
    .set_index("sample")
)

qc_test["Pass_D_SampleMAD"] = True

for sample in threshold_map.index:

    mask = (
        qc_test["sample"] == sample
    )

    lower_gene = threshold_map.loc[
        sample,
        "lower_genes_threshold"
    ]

    upper_mt = threshold_map.loc[
        sample,
        "upper_mt_threshold"
    ]

    qc_test.loc[
        mask,
        "Pass_D_SampleMAD"
    ] = (
        (
            qc_test.loc[
                mask,
                "n_genes_by_counts"
            ] >= lower_gene
        )
        &
        (
            qc_test.loc[
                mask,
                "pct_counts_mt"
            ] < upper_mt
        )
    )


mad_summary = (
    qc_test
    .groupby(
        "sample",
        observed=True
    )
    .agg(
        N_before=("sample", "size"),
        N_after=(
            "Pass_D_SampleMAD",
            "sum"
        )
    )
    .reset_index()
)

mad_summary["Retention_pct"] = (
    mad_summary["N_after"]
    /
    mad_summary["N_before"]
    * 100
)

display(mad_summary)

mad_summary.to_csv(
    RESULTS_DIR /
    "QC_SampleMAD_Retention.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP19 - QC THRESHOLD DECISION SUMMARY
# ============================================================

print("=" * 110)
print("STEP19 QC THRESHOLD DECISION SUMMARY")
print("=" * 110)


# ============================================================
# 1. Overall retention
# ============================================================

print("\n[1] OVERALL RETENTION UNDER FIXED QC RULES")
print("-" * 110)

display(
    overall_qc_sensitivity.round(3)
)


# ============================================================
# 2. Sample-level sensitivity
# ============================================================

print("\n[2] SAMPLE-LEVEL RETENTION UNDER FIXED QC RULES")
print("-" * 110)

sample_show_cols = [
    "sample",
    "N_before",

    "A_500genes_mt20_N",
    "A_500genes_mt20_Retention_pct",

    "B_600genes_mt15_N",
    "B_600genes_mt15_Retention_pct",

    "C_700genes_mt12_N",
    "C_700genes_mt12_Retention_pct"
]

display(
    sample_sensitivity[
        sample_show_cols
    ].round(2)
)


# ============================================================
# 3. Focus on potentially vulnerable samples
# ============================================================

print("\n[3] KEY SAMPLES TO WATCH")
print("-" * 110)

key_samples = [
    "HCC07T",
    "HCC10N",
    "HCC03N",
    "HCC08N"
]

key_sample_summary = (
    sample_sensitivity[
        sample_sensitivity["sample"].isin(
            key_samples
        )
    ][sample_show_cols]
    .copy()
)

display(
    key_sample_summary.round(2)
)


# ============================================================
# 4. Site-level retention
# ============================================================

print("\n[4] SITE-LEVEL RETENTION")
print("-" * 110)

display(
    site_sensitivity.round(2)
)


# ============================================================
# 5. Candidate B failure reasons
# ============================================================

print("\n[5] CANDIDATE B FAILURE REASONS")
print("-" * 110)

fail_low_genes = int(
    qc_test["Fail_low_genes_B"].sum()
)

fail_high_mt = int(
    qc_test["Fail_high_mt_B"].sum()
)

fail_both = int(
    qc_test["Fail_both_B"].sum()
)

fail_total = int(
    (~qc_test["Pass_B_600_15"]).sum()
)

print(
    f"Low genes (<600):        {fail_low_genes:,}"
)

print(
    f"High mt (>=15%):         {fail_high_mt:,}"
)

print(
    f"Both low genes + high mt:{fail_both:,}"
)

print(
    f"Total failing B:          {fail_total:,}"
)

print(
    f"Total cells:              {adata.n_obs:,}"
)

print(
    f"Overall removal rate:     "
    f"{fail_total / adata.n_obs * 100:.2f}%"
)


# ============================================================
# 6. Sample-specific MAD thresholds
# ============================================================

print("\n[6] SAMPLE-SPECIFIC MAD THRESHOLDS")
print("-" * 110)

mad_threshold_cols = [
    "sample",
    "median_genes",
    "MAD_genes",
    "lower_genes_threshold",
    "median_mt",
    "MAD_mt",
    "upper_mt_threshold"
]

display(
    sample_thresholds[
        mad_threshold_cols
    ].round(2)
)


# ============================================================
# 7. MAD-rule retention
# ============================================================

print("\n[7] SAMPLE-AWARE MAD RULE RETENTION")
print("-" * 110)

display(
    mad_summary.round(2)
)


# ============================================================
# 8. Focus on key samples under MAD rule
# ============================================================

print("\n[8] KEY SAMPLES UNDER MAD RULE")
print("-" * 110)

key_mad_summary = (
    mad_summary[
        mad_summary["sample"].isin(
            key_samples
        )
    ]
    .copy()
)

display(
    key_mad_summary.round(2)
)


# ============================================================
# 9. Overall MAD retention
# ============================================================

mad_total_before = int(
    mad_summary["N_before"].sum()
)

mad_total_after = int(
    mad_summary["N_after"].sum()
)

mad_removed = (
    mad_total_before -
    mad_total_after
)

mad_retention = (
    mad_total_after /
    mad_total_before *
    100
)

print("\n[9] OVERALL MAD RULE RESULT")
print("-" * 110)

print(
    "Cells before:",
    f"{mad_total_before:,}"
)

print(
    "Cells after:",
    f"{mad_total_after:,}"
)

print(
    "Cells removed:",
    f"{mad_removed:,}"
)

print(
    "Retention:",
    f"{mad_retention:.2f}%"
)


# ============================================================
# 10. Save compact decision tables
# ============================================================

key_sample_summary.to_csv(
    RESULTS_DIR /
    "QC_Key_Sample_Threshold_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

key_mad_summary.to_csv(
    RESULTS_DIR /
    "QC_Key_Sample_MAD_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 110)
print("END OF QC THRESHOLD DECISION SUMMARY")
print("=" * 110)

In [ ]:
# ============================================================
# STEP19 - FINAL CANDIDATE QC
# Sample-wise 5-MAD + absolute safety bounds
#
# IMPORTANT:
# This is still an AUDIT.
# No cells are actually removed in this cell.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 110)
print("STEP19 FINAL CANDIDATE QC AUDIT")
print("Sample-wise 5-MAD + absolute safety bounds")
print("=" * 110)


# ============================================================
# 0. Parameters
# ============================================================

N_MADS = 5

ABS_MIN_GENES = 200
ABS_MIN_COUNTS = 200
ABS_MAX_MT = 20.0


# ============================================================
# 1. MAD function
# ============================================================

def raw_mad(x):
    """
    Median Absolute Deviation.
    Uses unscaled/raw MAD.
    """
    x = np.asarray(x, dtype=float)

    med = np.median(x)

    return np.median(
        np.abs(x - med)
    )


# ============================================================
# 2. Prepare QC table
# ============================================================

qc_final = adata.obs.copy()

qc_final["log1p_total_counts"] = np.log1p(
    qc_final["total_counts"].astype(float)
)

qc_final["log1p_n_genes"] = np.log1p(
    qc_final["n_genes_by_counts"].astype(float)
)


# ============================================================
# 3. Calculate sample-specific 5-MAD thresholds
# ============================================================

threshold_rows = []

for sample, df in qc_final.groupby(
    "sample",
    observed=True
):

    # --------------------------------------------------------
    # log1p total counts
    # --------------------------------------------------------

    log_counts = df[
        "log1p_total_counts"
    ].to_numpy()

    med_log_counts = np.median(
        log_counts
    )

    mad_log_counts = raw_mad(
        log_counts
    )

    lower_log_counts = (
        med_log_counts
        -
        N_MADS * mad_log_counts
    )

    lower_counts_mad = np.expm1(
        lower_log_counts
    )

    # Add absolute safety floor
    effective_min_counts = max(
        ABS_MIN_COUNTS,
        lower_counts_mad
    )


    # --------------------------------------------------------
    # log1p detected genes
    # --------------------------------------------------------

    log_genes = df[
        "log1p_n_genes"
    ].to_numpy()

    med_log_genes = np.median(
        log_genes
    )

    mad_log_genes = raw_mad(
        log_genes
    )

    lower_log_genes = (
        med_log_genes
        -
        N_MADS * mad_log_genes
    )

    lower_genes_mad = np.expm1(
        lower_log_genes
    )

    # Add absolute safety floor
    effective_min_genes = max(
        ABS_MIN_GENES,
        lower_genes_mad
    )


    # --------------------------------------------------------
    # mitochondrial fraction
    # --------------------------------------------------------

    mt = df[
        "pct_counts_mt"
    ].to_numpy()

    med_mt = np.median(mt)
    mad_mt = raw_mad(mt)

    upper_mt_mad = (
        med_mt
        +
        N_MADS * mad_mt
    )

    # Absolute safety ceiling
    effective_max_mt = min(
        ABS_MAX_MT,
        upper_mt_mad
    )


    # --------------------------------------------------------
    # Record
    # --------------------------------------------------------

    threshold_rows.append({

        "sample":
            sample,

        "N_cells":
            len(df),

        "median_total_counts":
            np.median(
                df["total_counts"]
            ),

        "MAD_log1p_counts":
            mad_log_counts,

        "MAD_based_min_counts":
            lower_counts_mad,

        "effective_min_counts":
            effective_min_counts,

        "median_genes":
            np.median(
                df["n_genes_by_counts"]
            ),

        "MAD_log1p_genes":
            mad_log_genes,

        "MAD_based_min_genes":
            lower_genes_mad,

        "effective_min_genes":
            effective_min_genes,

        "median_mt":
            med_mt,

        "MAD_mt":
            mad_mt,

        "MAD_based_max_mt":
            upper_mt_mad,

        "effective_max_mt":
            effective_max_mt
    })


final_thresholds = pd.DataFrame(
    threshold_rows
)


# ============================================================
# 4. Apply thresholds as FLAGS ONLY
# ============================================================

threshold_map = (
    final_thresholds
    .set_index("sample")
)

qc_final["Fail_low_counts"] = False
qc_final["Fail_low_genes"] = False
qc_final["Fail_high_mt"] = False


for sample in threshold_map.index:

    mask = (
        qc_final["sample"] == sample
    )

    min_counts = threshold_map.loc[
        sample,
        "effective_min_counts"
    ]

    min_genes = threshold_map.loc[
        sample,
        "effective_min_genes"
    ]

    max_mt = threshold_map.loc[
        sample,
        "effective_max_mt"
    ]


    qc_final.loc[
        mask,
        "Fail_low_counts"
    ] = (
        qc_final.loc[
            mask,
            "total_counts"
        ] < min_counts
    )


    qc_final.loc[
        mask,
        "Fail_low_genes"
    ] = (
        qc_final.loc[
            mask,
            "n_genes_by_counts"
        ] < min_genes
    )


    qc_final.loc[
        mask,
        "Fail_high_mt"
    ] = (
        qc_final.loc[
            mask,
            "pct_counts_mt"
        ] >= max_mt
    )


# ============================================================
# 5. Final primary-QC flag
# ============================================================

qc_final["Pass_Final_5MAD"] = ~(
    qc_final["Fail_low_counts"]
    |
    qc_final["Fail_low_genes"]
    |
    qc_final["Fail_high_mt"]
)


# ============================================================
# 6. Overall summary
# ============================================================

N_before = len(qc_final)

N_after = int(
    qc_final[
        "Pass_Final_5MAD"
    ].sum()
)

N_removed = (
    N_before -
    N_after
)

retention_pct = (
    N_after /
    N_before *
    100
)

removal_pct = (
    N_removed /
    N_before *
    100
)

overall_final = pd.DataFrame(
    [{
        "Rule":
            "Sample-wise_5MAD_plus_safety_bounds",

        "N_before":
            N_before,

        "N_after":
            N_after,

        "N_removed":
            N_removed,

        "Retention_pct":
            retention_pct,

        "Removal_pct":
            removal_pct
    }]
)


# ============================================================
# 7. Sample-level retention
# ============================================================

sample_final = (
    qc_final
    .groupby(
        "sample",
        observed=True
    )
    .agg(
        N_before=(
            "sample",
            "size"
        ),

        N_after=(
            "Pass_Final_5MAD",
            "sum"
        )
    )
    .reset_index()
)

sample_final[
    "N_removed"
] = (
    sample_final["N_before"]
    -
    sample_final["N_after"]
)

sample_final[
    "Retention_pct"
] = (
    sample_final["N_after"]
    /
    sample_final["N_before"]
    *
    100
)

sample_final[
    "Removal_pct"
] = (
    100
    -
    sample_final[
        "Retention_pct"
    ]
)


# ============================================================
# 8. Site-level retention
# ============================================================

site_final = (
    qc_final
    .groupby(
        "site",
        observed=True
    )
    .agg(
        N_before=(
            "site",
            "size"
        ),

        N_after=(
            "Pass_Final_5MAD",
            "sum"
        )
    )
    .reset_index()
)

site_final[
    "N_removed"
] = (
    site_final["N_before"]
    -
    site_final["N_after"]
)

site_final[
    "Retention_pct"
] = (
    site_final["N_after"]
    /
    site_final["N_before"]
    *
    100
)

site_final[
    "Removal_pct"
] = (
    100
    -
    site_final[
        "Retention_pct"
    ]
)


# ============================================================
# 9. Failure reasons
# ============================================================

failure_summary = pd.DataFrame({

    "Reason": [
        "Low counts",
        "Low genes",
        "High mitochondrial fraction",
        "Any primary-QC failure"
    ],

    "N_cells": [

        int(
            qc_final[
                "Fail_low_counts"
            ].sum()
        ),

        int(
            qc_final[
                "Fail_low_genes"
            ].sum()
        ),

        int(
            qc_final[
                "Fail_high_mt"
            ].sum()
        ),

        int(
            (
                ~qc_final[
                    "Pass_Final_5MAD"
                ]
            ).sum()
        )
    ]
})

failure_summary[
    "Pct_of_total"
] = (
    failure_summary["N_cells"]
    /
    N_before
    *
    100
)


# ============================================================
# 10. Overlap of QC failure reasons
# ============================================================

qc_final[
    "N_QC_fail_reasons"
] = (
    qc_final[
        [
            "Fail_low_counts",
            "Fail_low_genes",
            "Fail_high_mt"
        ]
    ]
    .sum(axis=1)
)

failure_overlap = (
    qc_final[
        "N_QC_fail_reasons"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "N_fail_reasons"
    )
    .reset_index(
        name="N_cells"
    )
)

failure_overlap[
    "Pct_of_total"
] = (
    failure_overlap["N_cells"]
    /
    N_before
    *
    100
)


# ============================================================
# 11. Display results
# ============================================================

print("\n[1] FINAL SAMPLE-SPECIFIC THRESHOLDS")
print("-" * 110)

display(
    final_thresholds.round(2)
)


print("\n[2] OVERALL RETENTION")
print("-" * 110)

display(
    overall_final.round(3)
)


print("\n[3] SAMPLE-LEVEL RETENTION")
print("-" * 110)

display(
    sample_final.round(2)
)


print("\n[4] SITE-LEVEL RETENTION")
print("-" * 110)

display(
    site_final.round(2)
)


print("\n[5] FAILURE REASONS")
print("-" * 110)

display(
    failure_summary.round(3)
)


print("\n[6] FAILURE-REASON OVERLAP")
print("-" * 110)

display(
    failure_overlap.round(3)
)


# ============================================================
# 12. Highlight lowest-retention samples
# ============================================================

print("\n[7] LOWEST-RETENTION SAMPLES")
print("-" * 110)

lowest_samples = (
    sample_final
    .sort_values(
        "Retention_pct",
        ascending=True
    )
    .head(10)
)

display(
    lowest_samples.round(2)
)


# ============================================================
# 13. Save all audit results
# ============================================================

final_thresholds.to_csv(
    RESULTS_DIR /
    "QC_FinalCandidate_5MAD_Thresholds.csv",
    index=False,
    encoding="utf-8-sig"
)

overall_final.to_csv(
    RESULTS_DIR /
    "QC_FinalCandidate_5MAD_Overall.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_final.to_csv(
    RESULTS_DIR /
    "QC_FinalCandidate_5MAD_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)

site_final.to_csv(
    RESULTS_DIR /
    "QC_FinalCandidate_5MAD_BySite.csv",
    index=False,
    encoding="utf-8-sig"
)

failure_summary.to_csv(
    RESULTS_DIR /
    "QC_FinalCandidate_5MAD_FailureReasons.csv",
    index=False,
    encoding="utf-8-sig"
)

failure_overlap.to_csv(
    RESULTS_DIR /
    "QC_FinalCandidate_5MAD_FailureOverlap.csv",
    index=False,
    encoding="utf-8-sig"
)


# Save per-cell QC flags
qc_flags_output = qc_final[
    [
        "sample",
        "patient",
        "site",
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
        "Fail_low_counts",
        "Fail_low_genes",
        "Fail_high_mt",
        "N_QC_fail_reasons",
        "Pass_Final_5MAD"
    ]
].copy()

qc_flags_output.to_csv(
    RESULTS_DIR /
    "QC_FinalCandidate_5MAD_CellFlags.csv.gz",
    compression="gzip",
    encoding="utf-8"
)


print("\n" + "=" * 110)

print(
    f"Cells before : {N_before:,}"
)

print(
    f"Cells retained: {N_after:,}"
)

print(
    f"Cells removed : {N_removed:,}"
)

print(
    f"Retention     : {retention_pct:.2f}%"
)

print(
    f"Removal       : {removal_pct:.2f}%"
)

print("\nNO CELLS HAVE BEEN REMOVED YET.")

print("=" * 110)

In [ ]:
# ============================================================
# STEP19 - CELL-TYPE COMPOSITION BIAS AUDIT
#
# Purpose:
# Check whether the candidate 5-MAD QC rule disproportionately
# removes specific biological cell types.
#
# NOTE:
# "celltype" here is used ONLY as a QC reference annotation.
# It is NOT treated as our final downstream annotation.
# ============================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP19 CELL-TYPE COMPOSITION BIAS AUDIT")
print("=" * 110)


# ============================================================
# 1. Check metadata
# ============================================================

assert "celltype" in qc_final.columns, (
    "celltype column not found in qc_final."
)

assert "Pass_Final_5MAD" in qc_final.columns, (
    "Pass_Final_5MAD not found."
)

print("\nCell-type labels:")
print(
    qc_final["celltype"]
    .value_counts(dropna=False)
)


# ============================================================
# 2. Overall cell-type retention
# ============================================================

celltype_retention = (
    qc_final
    .groupby(
        "celltype",
        observed=True,
        dropna=False
    )
    .agg(
        N_before=(
            "celltype",
            "size"
        ),

        N_after=(
            "Pass_Final_5MAD",
            "sum"
        )
    )
    .reset_index()
)

celltype_retention[
    "N_removed"
] = (
    celltype_retention["N_before"]
    -
    celltype_retention["N_after"]
)

celltype_retention[
    "Retention_pct"
] = (
    celltype_retention["N_after"]
    /
    celltype_retention["N_before"]
    *
    100
)

celltype_retention[
    "Removal_pct"
] = (
    100
    -
    celltype_retention["Retention_pct"]
)

celltype_retention = (
    celltype_retention
    .sort_values(
        "Retention_pct",
        ascending=True
    )
)

print("\n[1] CELL-TYPE RETENTION")
print("-" * 110)

display(
    celltype_retention.round(2)
)


# ============================================================
# 3. Mitochondrial distribution by cell type
# ============================================================

celltype_mt = (
    qc_final
    .groupby(
        "celltype",
        observed=True,
        dropna=False
    )
    ["pct_counts_mt"]
    .agg(
        N="size",
        median="median",
        mean="mean",
        max="max",
        p90=lambda x:
            np.percentile(x, 90),
        p95=lambda x:
            np.percentile(x, 95),
        p99=lambda x:
            np.percentile(x, 99)
    )
    .reset_index()
)

celltype_mt = (
    celltype_mt
    .sort_values(
        "median",
        ascending=False
    )
)

print("\n[2] MITOCHONDRIAL FRACTION BY CELL TYPE")
print("-" * 110)

display(
    celltype_mt.round(2)
)


# ============================================================
# 4. Sample × cell-type retention
# ============================================================

sample_celltype_retention = (
    qc_final
    .groupby(
        [
            "sample",
            "celltype"
        ],
        observed=True,
        dropna=False
    )
    .agg(
        N_before=(
            "sample",
            "size"
        ),

        N_after=(
            "Pass_Final_5MAD",
            "sum"
        )
    )
    .reset_index()
)

sample_celltype_retention[
    "N_removed"
] = (
    sample_celltype_retention[
        "N_before"
    ]
    -
    sample_celltype_retention[
        "N_after"
    ]
)

sample_celltype_retention[
    "Retention_pct"
] = (
    sample_celltype_retention[
        "N_after"
    ]
    /
    sample_celltype_retention[
        "N_before"
    ]
    *
    100
)

sample_celltype_retention[
    "Removal_pct"
] = (
    100
    -
    sample_celltype_retention[
        "Retention_pct"
    ]
)


# ============================================================
# 5. Focus on combinations with enough cells
# ============================================================

sample_celltype_major = (
    sample_celltype_retention[
        sample_celltype_retention[
            "N_before"
        ] >= 50
    ]
    .sort_values(
        "Retention_pct",
        ascending=True
    )
)

print(
    "\n[3] SAMPLE × CELLTYPE RETENTION "
    "(groups with >=50 cells)"
)
print("-" * 110)

display(
    sample_celltype_major.round(2)
)


# ============================================================
# 6. Identify potentially concerning groups
# ============================================================

# This is a REVIEW FLAG, not an automatic exclusion rule.
concerning_groups = (
    sample_celltype_major[
        sample_celltype_major[
            "Retention_pct"
        ] < 85
    ]
    .copy()
)

print("\n[4] POTENTIALLY CONCERNING GROUPS")
print("Review flag: >=50 cells and retention <85%")
print("-" * 110)

if len(concerning_groups) == 0:

    print(
        "No major sample × cell-type group "
        "has retention <85%."
    )

else:

    display(
        concerning_groups.round(2)
    )


# ============================================================
# 7. Removed cells: cell-type composition
# ============================================================

removed_cells = qc_final[
    ~qc_final[
        "Pass_Final_5MAD"
    ]
].copy()

removed_celltypes = (
    removed_cells[
        "celltype"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "celltype"
    )
    .reset_index(
        name="N_removed"
    )
)

removed_celltypes[
    "Pct_of_removed"
] = (
    removed_celltypes[
        "N_removed"
    ]
    /
    len(removed_cells)
    *
    100
)

print("\n[5] COMPOSITION OF THE 5-MAD-REMOVED CELLS")
print("-" * 110)

display(
    removed_celltypes.round(2)
)


# ============================================================
# 8. Save outputs
# ============================================================

celltype_retention.to_csv(
    RESULTS_DIR /
    "QC_5MAD_CellType_Retention.csv",
    index=False,
    encoding="utf-8-sig"
)

celltype_mt.to_csv(
    RESULTS_DIR /
    "QC_5MAD_CellType_Mitochondrial_Distribution.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_celltype_retention.to_csv(
    RESULTS_DIR /
    "QC_5MAD_Sample_CellType_Retention.csv",
    index=False,
    encoding="utf-8-sig"
)

concerning_groups.to_csv(
    RESULTS_DIR /
    "QC_5MAD_Concerning_Sample_CellType_Groups.csv",
    index=False,
    encoding="utf-8-sig"
)

removed_celltypes.to_csv(
    RESULTS_DIR /
    "QC_5MAD_Removed_CellType_Composition.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 110)
print("AUDIT COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# STEP19 - LOCK PRIMARY QC DECISION
#
# Decision:
# GSE149614 is already an author-QC processed cohort.
# Secondary 5-MAD filtering showed biological composition bias,
# especially against hepatocytes.
#
# Therefore:
# - retain all 71,915 cells at this stage
# - preserve QC outlier flags for audit
# - proceed to sample-wise doublet detection
# ============================================================

import json
import pandas as pd
import numpy as np

print("=" * 100)
print("STEP19 PRIMARY QC DECISION")
print("=" * 100)

# Make sure cell order matches
assert adata.obs_names.equals(qc_final.index)

# Preserve secondary QC flags
adata.obs["qc_5mad_low_counts"] = (
    qc_final["Fail_low_counts"]
    .astype(bool)
    .values
)

adata.obs["qc_5mad_low_genes"] = (
    qc_final["Fail_low_genes"]
    .astype(bool)
    .values
)

adata.obs["qc_5mad_high_mt"] = (
    qc_final["Fail_high_mt"]
    .astype(bool)
    .values
)

adata.obs["qc_5mad_outlier"] = (
    ~qc_final["Pass_Final_5MAD"]
).astype(bool).values


# ------------------------------------------------------------
# Primary QC keep decision
# ------------------------------------------------------------

# IMPORTANT:
# No secondary hard filtering is performed because this cohort
# was already QC-filtered by the original study and our
# sensitivity analysis identified cell-type-dependent bias.
adata.obs["primary_qc_keep"] = True


print("Cells entering Step19:")
print(f"{adata.n_obs:,}")

print("\nCells hard-filtered by our secondary QC:")
print(0)

print("\nCells flagged by 5-MAD audit:")
print(
    f"{adata.obs['qc_5mad_outlier'].sum():,}"
)

print("\nCells retained for doublet detection:")
print(
    f"{adata.obs['primary_qc_keep'].sum():,}"
)


# ------------------------------------------------------------
# Save decision
# ------------------------------------------------------------

qc_decision = {

    "Dataset":
        "GSE149614",

    "Input_cells":
        int(adata.n_obs),

    "Dataset_status":
        "Processed cohort previously QC-filtered by original study",

    "Secondary_QC_methods_evaluated": [
        "Fixed threshold sensitivity analysis",
        "Sample-wise 4-MAD",
        "Sample-wise 5-MAD"
    ],

    "Secondary_5MAD_flagged_cells":
        int(
            adata.obs[
                "qc_5mad_outlier"
            ].sum()
        ),

    "Secondary_5MAD_hard_filter_applied":
        False,

    "Reason":
        (
            "Sample-wise mitochondrial filtering showed "
            "substantial cell-type-dependent depletion, "
            "particularly among hepatocytes; therefore "
            "5-MAD results are retained as audit flags only."
        ),

    "Cells_retained_before_doublet_detection":
        int(adata.n_obs),

    "Next_step":
        "Sample-wise Scrublet doublet detection"
}


with open(
    RESULTS_DIR /
    "Step19_Primary_QC_Decision.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        qc_decision,
        f,
        ensure_ascii=False,
        indent=4
    )


print("\nPRIMARY QC DECISION = LOCKED")
print("All cells retained before doublet detection.")
print("=" * 100)

In [ ]:
# ============================================================
# STEP19 - Save pre-doublet checkpoint
# ============================================================

PRE_DOUBLET_H5AD = (
    DATA_DIR /
    "GSE149614_PRE_DOUBLET_QC.h5ad"
)

adata.write_h5ad(
    PRE_DOUBLET_H5AD,
    compression="gzip"
)

print("Saved:")
print(PRE_DOUBLET_H5AD)

print(
    "Cells:",
    f"{adata.n_obs:,}"
)

print(
    "Genes:",
    f"{adata.n_vars:,}"
)

In [ ]:
import inspect
import scanpy as sc

print("Scanpy version:", sc.__version__)
print()
print(inspect.signature(sc.pp.scrublet))

In [ ]:
# ============================================================
# STEP19 - Install required Scrublet dependency
# ============================================================

import sys
import subprocess
import importlib.util

print("Current Python:")
print(sys.executable)

if importlib.util.find_spec("skimage") is None:

    print("\nscikit-image not found.")
    print("Installing scikit-image into the CURRENT kernel environment...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "scikit-image"
    ])

    print("\nInstallation finished.")

else:

    print("\nscikit-image is already installed.")

In [ ]:
import skimage
import scanpy as sc
import sys

print("=" * 80)
print("DEPENDENCY CHECK")
print("=" * 80)

print("Python:")
print(sys.executable)

print("\nScanpy:")
print(sc.__version__)

print("\nscikit-image:")
print(skimage.__version__)

print("\nSCRUBLET DEPENDENCY = PASS")

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP19 - Reload clean pre-doublet checkpoint
# ============================================================

from pathlib import Path
import scanpy as sc

PRE_DOUBLET_H5AD = Path(
    _hcc_path('LIHC_scRNA_Step19_QC/Data/GSE149614_PRE_DOUBLET_QC.h5ad')
)

assert PRE_DOUBLET_H5AD.exists()

adata = sc.read_h5ad(
    PRE_DOUBLET_H5AD
)

print("=" * 80)
print("PRE-DOUBLET CHECKPOINT RELOADED")
print("=" * 80)

print(adata)
print("Cells:", f"{adata.n_obs:,}")
print("Genes:", f"{adata.n_vars:,}")
print("Samples:", adata.obs["sample"].nunique())

print(
    "5-MAD audit flags:",
    f"{int(adata.obs['qc_5mad_outlier'].sum()):,}"
)

print(
    "Primary QC retained:",
    f"{int(adata.obs['primary_qc_keep'].sum()):,}"
)

In [ ]:
# ============================================================
# STEP19 - SAMPLE-WISE DOUBLET DETECTION
# Scrublet / Scanpy 1.11.4
# ============================================================

import scanpy as sc
import numpy as np
import time

print("=" * 100)
print("STEP19 SAMPLE-WISE SCRUBLET")
print("=" * 100)

print("Scanpy version:", sc.__version__)
print("Cells:", f"{adata.n_obs:,}")
print("Samples:", adata.obs["sample"].nunique())


# ------------------------------------------------------------
# Raw-count check
# ------------------------------------------------------------

if hasattr(adata.X, "data"):
    sampled_values = adata.X[:1000, :1000].data
else:
    sampled_values = np.asarray(
        adata.X[:1000, :1000]
    ).ravel()

if len(sampled_values) > 0:

    integer_fraction = np.mean(
        np.isclose(
            sampled_values,
            np.round(sampled_values)
        )
    )

else:
    integer_fraction = 1.0


print(
    "Approx. integer fraction:",
    f"{integer_fraction:.4f}"
)

assert integer_fraction > 0.99

print("Raw-count input check = PASS")


# ------------------------------------------------------------
# Run Scrublet
# ------------------------------------------------------------

print("\nStarting Scrublet...")
print(
    "Each of the 21 samples will be "
    "handled independently."
)

start_time = time.time()

sc.pp.scrublet(
    adata,

    batch_key="sample",

    sim_doublet_ratio=2.0,

    expected_doublet_rate=0.05,

    n_prin_comps=30,

    random_state=42,

    verbose=True
)

elapsed = (
    time.time() -
    start_time
) / 60


# ------------------------------------------------------------
# Verify result
# ------------------------------------------------------------

required_cols = [
    "doublet_score",
    "predicted_doublet"
]

for col in required_cols:
    assert col in adata.obs.columns


n_doublet = int(
    adata.obs[
        "predicted_doublet"
    ].sum()
)

doublet_rate = (
    n_doublet /
    adata.n_obs *
    100
)


print("\n" + "=" * 100)
print("SCRUBLET COMPLETE")
print("=" * 100)

print(
    "Elapsed:",
    f"{elapsed:.2f} min"
)

print(
    "Predicted doublets:",
    f"{n_doublet:,}"
)

print(
    "Predicted doublet rate:",
    f"{doublet_rate:.2f}%"
)

print("=" * 100)

In [ ]:
# ============================================================
# STEP19 - SCRUBLET FULL AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP19 SCRUBLET FULL AUDIT")
print("=" * 110)


# ============================================================
# 1. Overall result
# ============================================================

n_total = adata.n_obs

n_doublet = int(
    adata.obs["predicted_doublet"].sum()
)

doublet_rate = (
    n_doublet /
    n_total *
    100
)

print("\n[1] OVERALL")
print("-" * 110)

print(
    "Total cells:",
    f"{n_total:,}"
)

print(
    "Predicted doublets:",
    f"{n_doublet:,}"
)

print(
    "Predicted doublet rate:",
    f"{doublet_rate:.3f}%"
)


# ============================================================
# 2. Doublets by sample
# ============================================================

doublet_by_sample = (
    adata.obs
    .groupby(
        "sample",
        observed=True
    )
    .agg(
        N_cells=(
            "sample",
            "size"
        ),

        N_doublet=(
            "predicted_doublet",
            "sum"
        ),

        median_doublet_score=(
            "doublet_score",
            "median"
        ),

        mean_doublet_score=(
            "doublet_score",
            "mean"
        ),

        p95_doublet_score=(
            "doublet_score",
            lambda x:
                np.percentile(x, 95)
        ),

        p99_doublet_score=(
            "doublet_score",
            lambda x:
                np.percentile(x, 99)
        ),

        max_doublet_score=(
            "doublet_score",
            "max"
        )
    )
    .reset_index()
)

doublet_by_sample[
    "Doublet_pct"
] = (
    doublet_by_sample["N_doublet"]
    /
    doublet_by_sample["N_cells"]
    *
    100
)

print("\n[2] DOUBLETS BY SAMPLE")
print("-" * 110)

display(
    doublet_by_sample.round(4)
)


# ============================================================
# 3. Doublets by original cell type
# ============================================================

doublet_by_celltype = (
    adata.obs
    .groupby(
        "celltype",
        observed=True
    )
    .agg(
        N_cells=(
            "celltype",
            "size"
        ),

        N_doublet=(
            "predicted_doublet",
            "sum"
        ),

        median_doublet_score=(
            "doublet_score",
            "median"
        ),

        mean_doublet_score=(
            "doublet_score",
            "mean"
        )
    )
    .reset_index()
)

doublet_by_celltype[
    "Doublet_pct"
] = (
    doublet_by_celltype["N_doublet"]
    /
    doublet_by_celltype["N_cells"]
    *
    100
)

print("\n[3] DOUBLETS BY ORIGINAL CELLTYPE")
print("-" * 110)

display(
    doublet_by_celltype
    .sort_values(
        "Doublet_pct",
        ascending=False
    )
    .round(4)
)


# ============================================================
# 4. Singlet vs predicted-doublet QC characteristics
# ============================================================

doublet_qc = (
    adata.obs
    .groupby(
        "predicted_doublet",
        observed=True
    )
    .agg(
        N=(
            "predicted_doublet",
            "size"
        ),

        median_genes=(
            "n_genes_by_counts",
            "median"
        ),

        mean_genes=(
            "n_genes_by_counts",
            "mean"
        ),

        median_counts=(
            "total_counts",
            "median"
        ),

        mean_counts=(
            "total_counts",
            "mean"
        ),

        median_mt=(
            "pct_counts_mt",
            "median"
        ),

        median_doublet_score=(
            "doublet_score",
            "median"
        ),

        mean_doublet_score=(
            "doublet_score",
            "mean"
        )
    )
    .reset_index()
)

print("\n[4] SINGLET vs PREDICTED DOUBLET")
print("-" * 110)

display(
    doublet_qc.round(3)
)


# ============================================================
# 5. Doublet-score quantiles
# ============================================================

score_quantiles = (
    adata.obs[
        "doublet_score"
    ]
    .quantile(
        [
            0,
            0.50,
            0.75,
            0.90,
            0.95,
            0.975,
            0.99,
            0.995,
            0.999,
            1
        ]
    )
)

score_quantiles = (
    score_quantiles
    .rename(
        "doublet_score"
    )
    .to_frame()
)

print("\n[5] DOUBLET SCORE QUANTILES")
print("-" * 110)

display(
    score_quantiles.round(5)
)


# ============================================================
# 6. Interaction with previous 5-MAD QC flag
# ============================================================

if "qc_5mad_outlier" in adata.obs.columns:

    doublet_vs_qc = pd.crosstab(
        adata.obs[
            "qc_5mad_outlier"
        ],
        adata.obs[
            "predicted_doublet"
        ],
        margins=True
    )

    print("\n[6] 5-MAD FLAG × SCRUBLET")
    print("-" * 110)

    display(
        doublet_vs_qc
    )


# ============================================================
# 7. Examine predicted doublets directly
# ============================================================

predicted_doublets = (
    adata.obs.loc[
        adata.obs[
            "predicted_doublet"
        ]
    ]
    [
        [
            "sample",
            "patient",
            "site",
            "celltype",
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
            "doublet_score"
        ]
    ]
    .copy()
)

predicted_doublets = (
    predicted_doublets
    .sort_values(
        "doublet_score",
        ascending=False
    )
)

print("\n[7] TOP 30 PREDICTED DOUBLETS")
print("-" * 110)

display(
    predicted_doublets
    .head(30)
    .round(4)
)


# ============================================================
# 8. Check Scanpy/Scrublet metadata
# ============================================================

print("\n[8] SCRUBLET UNS METADATA")
print("-" * 110)

if "scrublet" in adata.uns:

    print(
        "Top-level keys:",
        list(
            adata.uns[
                "scrublet"
            ].keys()
        )
    )

    # With batch_key Scanpy often stores
    # sample-specific information under batches.
    if "batches" in adata.uns["scrublet"]:

        print("\nNumber of Scrublet batches:")
        print(
            len(
                adata.uns[
                    "scrublet"
                ][
                    "batches"
                ]
            )
        )

        print("\nExample batch keys:")
        print(
            list(
                adata.uns[
                    "scrublet"
                ][
                    "batches"
                ].keys()
            )[:10]
        )

else:

    print(
        "No adata.uns['scrublet'] "
        "metadata found."
    )


# ============================================================
# 9. Save audit outputs
# ============================================================

doublet_by_sample.to_csv(
    RESULTS_DIR /
    "Scrublet_By_Sample.csv",
    index=False,
    encoding="utf-8-sig"
)

doublet_by_celltype.to_csv(
    RESULTS_DIR /
    "Scrublet_By_CellType.csv",
    index=False,
    encoding="utf-8-sig"
)

doublet_qc.to_csv(
    RESULTS_DIR /
    "Scrublet_Singlet_vs_Doublet_QC.csv",
    index=False,
    encoding="utf-8-sig"
)

score_quantiles.to_csv(
    RESULTS_DIR /
    "Scrublet_DoubletScore_Quantiles.csv",
    encoding="utf-8-sig"
)

predicted_doublets.to_csv(
    RESULTS_DIR /
    "Scrublet_Predicted_Doublets.csv",
    encoding="utf-8-sig"
)


print("\n" + "=" * 110)
print("SCRUBLET AUDIT COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# STEP19 - SAVE FULL SCRUBLET CHECKPOINT
# 71,915 cells, no doublets removed yet
# ============================================================

from pathlib import Path

SCRUBLET_FULL_H5AD = (
    DATA_DIR /
    "GSE149614_SCRUBLET_ANNOTATED_ALL_CELLS.h5ad"
)

adata.write_h5ad(
    SCRUBLET_FULL_H5AD,
    compression="gzip"
)

print("=" * 90)
print("FULL SCRUBLET CHECKPOINT SAVED")
print("=" * 90)

print("Path:")
print(SCRUBLET_FULL_H5AD)

print("\nCells:")
print(f"{adata.n_obs:,}")

print("Genes:")
print(f"{adata.n_vars:,}")

print("Predicted doublets:")
print(
    f"{int(adata.obs['predicted_doublet'].sum()):,}"
)

print(
    "File size:",
    f"{SCRUBLET_FULL_H5AD.stat().st_size / 1024**3:.3f} GB"
)

In [ ]:
# ============================================================
# STEP19 - SAVE PREDICTED DOUBLET CELL LIST
# ============================================================

doublet_cells = (
    adata.obs.loc[
        adata.obs["predicted_doublet"]
    ]
    [
        [
            "sample",
            "patient",
            "site",
            "celltype",
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
            "doublet_score",
            "qc_5mad_outlier"
        ]
    ]
    .copy()
)

doublet_cells.index.name = "Cell"

doublet_cells = (
    doublet_cells
    .sort_values(
        "doublet_score",
        ascending=False
    )
)

DOUBLET_LIST_PATH = (
    RESULTS_DIR /
    "Scrublet_Removed_Doublet_Barcodes.csv"
)

doublet_cells.to_csv(
    DOUBLET_LIST_PATH,
    encoding="utf-8-sig"
)

print("Saved:")
print(DOUBLET_LIST_PATH)

print("\nNumber of cells:")
print(f"{len(doublet_cells):,}")

display(
    doublet_cells.head(20)
)

In [ ]:
# ============================================================
# STEP19 - SAVE SCRUBLET SCORE DISTRIBUTION
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.hist(
    adata.obs.loc[
        ~adata.obs["predicted_doublet"],
        "doublet_score"
    ],
    bins=80,
    alpha=0.7,
    label="Predicted singlet"
)

ax.hist(
    adata.obs.loc[
        adata.obs["predicted_doublet"],
        "doublet_score"
    ],
    bins=40,
    alpha=0.7,
    label="Predicted doublet"
)

ax.set_xlabel(
    "Scrublet doublet score"
)

ax.set_ylabel(
    "Number of cells"
)

ax.set_title(
    "Scrublet doublet-score distribution"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "Scrublet_DoubletScore_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP19 - SAVE SAMPLE DOUBLET RATE FIGURE
# ============================================================

plot_df = (
    doublet_by_sample
    .sort_values(
        "Doublet_pct",
        ascending=False
    )
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.bar(
    plot_df["sample"],
    plot_df["Doublet_pct"]
)

ax.set_xlabel("Sample")
ax.set_ylabel("Predicted doublets (%)")

ax.set_title(
    "Scrublet-predicted doublet rate by sample"
)

ax.tick_params(
    axis="x",
    rotation=60
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "Scrublet_DoubletRate_BySample.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP19 - LOCK DOUBLET DECISION
# ============================================================

import json

adata.obs["scrublet_keep"] = (
    ~adata.obs["predicted_doublet"]
)

N_BEFORE = int(
    adata.n_obs
)

N_DOUBLET = int(
    adata.obs[
        "predicted_doublet"
    ].sum()
)

N_AFTER = int(
    adata.obs[
        "scrublet_keep"
    ].sum()
)

DOUBLET_RATE = (
    N_DOUBLET /
    N_BEFORE *
    100
)

doublet_decision = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step19_scRNA_QC",

    "Doublet_method":
        "Scanpy Scrublet",

    "Scanpy_version":
        str(sc.__version__),

    "Doublet_detection_mode":
        "sample-wise",

    "Batch_key":
        "sample",

    "Number_of_samples":
        int(
            adata.obs[
                "sample"
            ].nunique()
        ),

    "Input_cells":
        N_BEFORE,

    "Predicted_doublets":
        N_DOUBLET,

    "Predicted_doublet_rate_pct":
        float(
            DOUBLET_RATE
        ),

    "Doublet_removal":
        True,

    "Cells_after_doublet_removal":
        N_AFTER,

    "Secondary_5MAD_hard_filter":
        False,

    "Secondary_5MAD_policy":
        (
            "Retained as QC audit flags only "
            "because hard mitochondrial filtering "
            "caused substantial cell-type-dependent "
            "depletion, particularly in hepatocytes."
        ),

    "Final_Step19_policy":
        (
            "Retain author-QC cells, do not apply "
            "secondary mitochondrial hard filtering, "
            "remove only Scrublet-predicted doublets."
        ),

    "Next_step":
        "Step20_Normalization_Integration"
}


DOUBLET_DECISION_PATH = (
    RESULTS_DIR /
    "Step19_Doublet_Decision.json"
)

with open(
    DOUBLET_DECISION_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        doublet_decision,
        f,
        ensure_ascii=False,
        indent=4
    )

print(
    json.dumps(
        doublet_decision,
        ensure_ascii=False,
        indent=4
    )
)

In [ ]:
# ============================================================
# STEP19 - REMOVE SCRUBLET-PREDICTED DOUBLETS
# ============================================================

adata_clean = (
    adata[
        adata.obs[
            "scrublet_keep"
        ]
    ]
    .copy()
)

print("=" * 90)
print("DOUBLET REMOVAL")
print("=" * 90)

print(
    "Before:",
    f"{adata.n_obs:,}"
)

print(
    "Removed:",
    f"{adata.n_obs - adata_clean.n_obs:,}"
)

print(
    "After:",
    f"{adata_clean.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_clean.n_vars:,}"
)

assert adata_clean.n_obs == 71782

assert (
    adata_clean.obs[
        "predicted_doublet"
    ].sum()
    == 0
)

print("\nDOUBLET REMOVAL = PASS")

In [ ]:
# ============================================================
# STEP19 - BEFORE / AFTER DOUBLET RETENTION AUDIT
# ============================================================

def retention_table(
    obs,
    group_col,
    keep_col="scrublet_keep"
):

    out = (
        obs
        .groupby(
            group_col,
            observed=True
        )
        .agg(
            N_before=(
                group_col,
                "size"
            ),

            N_after=(
                keep_col,
                "sum"
            )
        )
        .reset_index()
    )

    out["N_removed"] = (
        out["N_before"]
        -
        out["N_after"]
    )

    out["Retention_pct"] = (
        out["N_after"]
        /
        out["N_before"]
        *
        100
    )

    return out


sample_retention_final = retention_table(
    adata.obs,
    "sample"
)

site_retention_final = retention_table(
    adata.obs,
    "site"
)

celltype_retention_final = retention_table(
    adata.obs,
    "celltype"
)


print("\n[1] SAMPLE RETENTION")
display(
    sample_retention_final.round(3)
)

print("\n[2] SITE RETENTION")
display(
    site_retention_final.round(3)
)

print("\n[3] ORIGINAL CELLTYPE RETENTION")
display(
    celltype_retention_final.round(3)
)


sample_retention_final.to_csv(
    RESULTS_DIR /
    "Step19_Final_Retention_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)

site_retention_final.to_csv(
    RESULTS_DIR /
    "Step19_Final_Retention_BySite.csv",
    index=False,
    encoding="utf-8-sig"
)

celltype_retention_final.to_csv(
    RESULTS_DIR /
    "Step19_Final_Retention_ByOriginalCellType.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP19 - SAVE FINAL CLEAN RAW COUNTS
# ============================================================

CLEAN_H5AD = (
    RESULTS_DIR /
    "GSE149614_CLEAN_COUNTS.h5ad"
)

adata_clean.write_h5ad(
    CLEAN_H5AD,
    compression="gzip"
)

print("=" * 90)
print("FINAL CLEAN COUNTS SAVED")
print("=" * 90)

print("Path:")
print(CLEAN_H5AD)

print("\nShape:")
print(adata_clean.shape)

print(
    "\nCells:",
    f"{adata_clean.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_clean.n_vars:,}"
)

print(
    "\nFile size:",
    f"{CLEAN_H5AD.stat().st_size / 1024**3:.3f} GB"
)

In [ ]:
# ============================================================
# STEP19 - FINAL AUDIT MANIFEST
# ============================================================

import json

step19_final_audit = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step19_scRNA_QC",

    "Status":
        "COMPLETE",

    "Initial_cells":
        71915,

    "Initial_genes":
        25712,

    "Author_processed_QC_cohort":
        True,

    "Secondary_QC_metrics_recalculated":
        True,

    "Fixed_threshold_sensitivity_tested":
        True,

    "Sample_wise_MAD_sensitivity_tested":
        True,

    "Secondary_mt_hard_filter_applied":
        False,

    "Reason_no_secondary_mt_filter":
        (
            "Sample-wise mitochondrial filtering "
            "showed cell-type-dependent depletion, "
            "especially hepatocytes."
        ),

    "Scrublet_run":
        True,

    "Scrublet_mode":
        "sample-wise",

    "Scrublet_samples":
        21,

    "Predicted_doublets_removed":
        133,

    "Predicted_doublet_rate_pct":
        133 / 71915 * 100,

    "Final_cells":
        int(
            adata_clean.n_obs
        ),

    "Final_genes":
        int(
            adata_clean.n_vars
        ),

    "Expression_matrix":
        "raw integer counts",

    "Normalization_performed":
        False,

    "Integration_performed":
        False,

    "Output_clean_h5ad":
        str(
            CLEAN_H5AD
        ),

    "Full_pre_filter_checkpoint":
        str(
            SCRUBLET_FULL_H5AD
        ),

    "Next_step":
        "Step20_Normalization_Integration"
}


FINAL_AUDIT_PATH = (
    RESULTS_DIR /
    "Step19_Final_Audit.json"
)

with open(
    FINAL_AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step19_final_audit,
        f,
        ensure_ascii=False,
        indent=4
    )

print(
    json.dumps(
        step19_final_audit,
        ensure_ascii=False,
        indent=4
    )
)

In [ ]:
# ============================================================
# STEP19 - FINAL CHECKPOINT
# ============================================================

print("=" * 100)

print("STEP19 STATUS = COMPLETE")

print("\nDATASET:")
print("GSE149614")

print("\nINITIAL:")
print("71,915 cells × 25,712 genes")

print("\nSECONDARY QC:")
print(
    "No mitochondrial / gene-count "
    "hard filtering applied"
)

print(
    "5-MAD retained as audit flags only"
)

print("\nDOUBLET:")
print("Method = sample-wise Scrublet")

print(
    "Removed =",
    f"{133:,}"
)

print(
    "Rate =",
    f"{133 / 71915 * 100:.3f}%"
)

print("\nFINAL CLEAN COUNTS:")

print(
    f"{adata_clean.n_obs:,} cells × "
    f"{adata_clean.n_vars:,} genes"
)

print("\nRAW COUNTS = PRESERVED")

print("NORMALIZATION = NOT PERFORMED")
print("INTEGRATION = NOT PERFORMED")

print("\nNEXT:")
print(
    "Step20 — Normalization / Integration"
)

print("=" * 100)

In [ ]:
# ============================================================
# STEP19 - FINAL RESULTS / FIGURES EXPORT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from datetime import datetime

print("=" * 100)
print("STEP19 FINAL EXPORT")
print("=" * 100)

# ------------------------------------------------------------
# Directories
# ------------------------------------------------------------

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. CORE QC TABLES
# ============================================================

print("\n[1] Saving core QC tables...")

# Global QC
qc_summary.to_csv(
    RESULTS_DIR / "QC_Global_Summary.csv",
    encoding="utf-8-sig"
)

# Sample QC before filtering
sample_qc.to_csv(
    RESULTS_DIR / "QC_By_Sample_Before_Filtering.csv",
    index=False,
    encoding="utf-8-sig"
)

# Patient QC
patient_qc.to_csv(
    RESULTS_DIR / "QC_By_Patient_Before_Filtering.csv",
    index=False,
    encoding="utf-8-sig"
)

# Site QC
site_qc.to_csv(
    RESULTS_DIR / "QC_By_Site_Before_Filtering.csv",
    index=False,
    encoding="utf-8-sig"
)

# Quantiles
quantile_table.to_csv(
    RESULTS_DIR / "QC_Quantiles.csv",
    encoding="utf-8-sig"
)


# ============================================================
# 2. THRESHOLD SENSITIVITY TABLES
# ============================================================

print("[2] Saving QC threshold sensitivity tables...")

overall_qc_sensitivity.to_csv(
    RESULTS_DIR / "QC_Threshold_Sensitivity_Overall.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_sensitivity.to_csv(
    RESULTS_DIR / "QC_Threshold_Sensitivity_By_Sample.csv",
    index=False,
    encoding="utf-8-sig"
)

site_sensitivity.to_csv(
    RESULTS_DIR / "QC_Threshold_Sensitivity_By_Site.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_thresholds.to_csv(
    RESULTS_DIR / "QC_SampleSpecific_MAD_Thresholds.csv",
    index=False,
    encoding="utf-8-sig"
)

mad_summary.to_csv(
    RESULTS_DIR / "QC_4MAD_Retention_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)

final_thresholds.to_csv(
    RESULTS_DIR / "QC_5MAD_FinalCandidate_Thresholds.csv",
    index=False,
    encoding="utf-8-sig"
)

overall_final.to_csv(
    RESULTS_DIR / "QC_5MAD_FinalCandidate_Overall.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_final.to_csv(
    RESULTS_DIR / "QC_5MAD_FinalCandidate_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)

site_final.to_csv(
    RESULTS_DIR / "QC_5MAD_FinalCandidate_BySite.csv",
    index=False,
    encoding="utf-8-sig"
)

failure_summary.to_csv(
    RESULTS_DIR / "QC_5MAD_FailureReasons.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 3. CELL-TYPE BIAS AUDIT
# ============================================================

print("[3] Saving cell-type bias audit tables...")

celltype_retention.to_csv(
    RESULTS_DIR / "QC_5MAD_CellType_Retention.csv",
    index=False,
    encoding="utf-8-sig"
)

celltype_mt.to_csv(
    RESULTS_DIR / "QC_5MAD_CellType_Mitochondrial_Distribution.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_celltype_retention.to_csv(
    RESULTS_DIR / "QC_5MAD_Sample_CellType_Retention.csv",
    index=False,
    encoding="utf-8-sig"
)

concerning_groups.to_csv(
    RESULTS_DIR / "QC_5MAD_Concerning_Sample_CellType_Groups.csv",
    index=False,
    encoding="utf-8-sig"
)

removed_celltypes.to_csv(
    RESULTS_DIR / "QC_5MAD_Flagged_CellType_Composition.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 4. SCRUBLET TABLES
# ============================================================

print("[4] Saving Scrublet tables...")

doublet_by_sample.to_csv(
    RESULTS_DIR / "Scrublet_By_Sample.csv",
    index=False,
    encoding="utf-8-sig"
)

doublet_by_celltype.to_csv(
    RESULTS_DIR / "Scrublet_By_CellType.csv",
    index=False,
    encoding="utf-8-sig"
)

doublet_qc.to_csv(
    RESULTS_DIR / "Scrublet_Singlet_vs_Doublet_QC.csv",
    index=False,
    encoding="utf-8-sig"
)

score_quantiles.to_csv(
    RESULTS_DIR / "Scrublet_DoubletScore_Quantiles.csv",
    encoding="utf-8-sig"
)

predicted_doublets.to_csv(
    RESULTS_DIR / "Scrublet_Predicted_Doublets_All.csv",
    encoding="utf-8-sig"
)

doublet_cells.to_csv(
    RESULTS_DIR / "Scrublet_Removed_133_Doublet_Barcodes.csv",
    encoding="utf-8-sig"
)


# ============================================================
# 5. FINAL RETENTION TABLES
# ============================================================

print("[5] Saving final retention tables...")

sample_retention_final.to_csv(
    RESULTS_DIR / "Step19_Final_Retention_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)

site_retention_final.to_csv(
    RESULTS_DIR / "Step19_Final_Retention_BySite.csv",
    index=False,
    encoding="utf-8-sig"
)

celltype_retention_final.to_csv(
    RESULTS_DIR / "Step19_Final_Retention_ByOriginalCellType.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 6. FIGURE: QC distributions
# ============================================================

print("[6] Saving QC figures...")

# -----------------------------
# Genes per cell
# -----------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    adata.obs["n_genes_by_counts"],
    bins=100
)

ax.set_xlabel("Genes detected per cell")
ax.set_ylabel("Number of cells")
ax.set_title("Detected genes per cell")

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "QC_01_nGenes_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# -----------------------------
# Total counts
# -----------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    np.log10(
        adata.obs["total_counts"] + 1
    ),
    bins=100
)

ax.set_xlabel("log10(total counts + 1)")
ax.set_ylabel("Number of cells")
ax.set_title("Total counts per cell")

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "QC_02_TotalCounts_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# -----------------------------
# Mitochondrial %
# -----------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    adata.obs["pct_counts_mt"],
    bins=100
)

ax.set_xlabel("Mitochondrial counts (%)")
ax.set_ylabel("Number of cells")
ax.set_title("Mitochondrial fraction")

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "QC_03_Mitochondrial_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 7. FIGURE: genes vs counts
# ============================================================

rng = np.random.default_rng(42)

plot_n = min(
    30000,
    adata.n_obs
)

plot_idx = rng.choice(
    adata.n_obs,
    size=plot_n,
    replace=False
)

plot_obs = adata.obs.iloc[
    plot_idx
]

fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(
    np.log10(
        plot_obs["total_counts"] + 1
    ),
    plot_obs["n_genes_by_counts"],
    s=4,
    alpha=0.25
)

ax.set_xlabel(
    "log10(total counts + 1)"
)

ax.set_ylabel(
    "Genes detected"
)

ax.set_title(
    "Detected genes vs total counts"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "QC_04_Genes_vs_Counts.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 8. FIGURE: Scrublet score
# ============================================================

print("[7] Saving Scrublet figures...")

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    adata.obs.loc[
        ~adata.obs["predicted_doublet"],
        "doublet_score"
    ],
    bins=80,
    alpha=0.7,
    label="Predicted singlet"
)

ax.hist(
    adata.obs.loc[
        adata.obs["predicted_doublet"],
        "doublet_score"
    ],
    bins=40,
    alpha=0.7,
    label="Predicted doublet"
)

ax.set_xlabel(
    "Scrublet doublet score"
)

ax.set_ylabel(
    "Number of cells"
)

ax.set_title(
    "Scrublet doublet-score distribution"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "QC_05_Scrublet_DoubletScore.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 9. FIGURE: Doublet rate by sample
# ============================================================

plot_df = (
    doublet_by_sample
    .sort_values(
        "Doublet_pct",
        ascending=False
    )
)

fig, ax = plt.subplots(figsize=(11, 6))

ax.bar(
    plot_df["sample"],
    plot_df["Doublet_pct"]
)

ax.set_xlabel("Sample")
ax.set_ylabel("Predicted doublets (%)")

ax.set_title(
    "Scrublet-predicted doublet rate by sample"
)

ax.tick_params(
    axis="x",
    rotation=60
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "QC_06_Scrublet_DoubletRate_BySample.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 10. FIGURE: final retention by sample
# ============================================================

plot_ret = (
    sample_retention_final
    .sort_values(
        "Retention_pct",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(
    plot_ret["sample"],
    plot_ret["Retention_pct"]
)

ax.set_xlabel(
    "Cells retained after doublet removal (%)"
)

ax.set_ylabel(
    "Sample"
)

ax.set_title(
    "Final Step19 cell retention by sample"
)

ax.set_xlim(
    98,
    100.1
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "QC_07_Final_Retention_BySample.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 11. FIGURE: cell-type doublet rates
# ============================================================

plot_ct = (
    doublet_by_celltype
    .sort_values(
        "Doublet_pct",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    plot_ct["celltype"],
    plot_ct["Doublet_pct"]
)

ax.set_xlabel(
    "Predicted doublets (%)"
)

ax.set_ylabel(
    "Original cell type"
)

ax.set_title(
    "Scrublet doublet rate by original cell type"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "QC_08_Scrublet_DoubletRate_ByCellType.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 12. SAVE FINAL STEP19 DECISION SUMMARY
# ============================================================

step19_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step19_scRNA_QC",

    "Status":
        "COMPLETE",

    "Initial_cells":
        int(adata.n_obs),

    "Initial_genes":
        int(adata.n_vars),

    "Secondary_hard_QC":
        False,

    "Reason_secondary_QC_not_applied":
        (
            "Sample-wise mitochondrial filtering caused "
            "substantial cell-type-dependent depletion, "
            "especially among hepatocytes."
        ),

    "QC_outlier_policy":
        "5-MAD retained as audit flags only",

    "Doublet_method":
        "Sample-wise Scrublet",

    "Scrublet_batches":
        int(
            adata.obs[
                "sample"
            ].nunique()
        ),

    "Predicted_doublets_removed":
        int(
            adata.obs[
                "predicted_doublet"
            ].sum()
        ),

    "Predicted_doublet_rate_pct":
        float(
            adata.obs[
                "predicted_doublet"
            ].mean() * 100
        ),

    "Final_cells":
        int(
            adata_clean.n_obs
        ),

    "Final_genes":
        int(
            adata_clean.n_vars
        ),

    "Raw_counts_preserved":
        True,

    "Normalization_performed":
        False,

    "Integration_performed":
        False,

    "Final_clean_file":
        str(
            CLEAN_H5AD
        )
}


with open(
    RESULTS_DIR /
    "Step19_Final_Summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step19_summary,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 13. CREATE FILE MANIFEST
# ============================================================

result_files = sorted(
    [
        p
        for p in RESULTS_DIR.glob("*")
        if p.is_file()
    ]
)

figure_files = sorted(
    [
        p
        for p in FIGURES_DIR.glob("*")
        if p.is_file()
    ]
)

manifest_rows = []

for p in result_files:

    manifest_rows.append({
        "Category":
            "Result",

        "Filename":
            p.name,

        "Size_MB":
            round(
                p.stat().st_size /
                1024**2,
                3
            )
    })

for p in figure_files:

    manifest_rows.append({
        "Category":
            "Figure",

        "Filename":
            p.name,

        "Size_MB":
            round(
                p.stat().st_size /
                1024**2,
                3
            )
    })


manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_df.to_csv(
    RESULTS_DIR /
    "Step19_Output_Manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 100)
print("STEP19 FINAL EXPORT COMPLETE")
print("=" * 100)

print(
    "Results files:",
    len(result_files)
)

print(
    "Figure files:",
    len(figure_files)
)

print("\nFinal clean object:")
print(CLEAN_H5AD)

print("\nManifest:")
print(
    RESULTS_DIR /
    "Step19_Output_Manifest.csv"
)

print("=" * 100)